In [1]:
import pyclifford as pc
import mcp_server as mcp
import numpy
from math import ceil

def sample_random_pauli_string(N):
    """Sample a random Pauli operator string with random phase for N qubits.
    Returns a string like '-iIXIIYZXIZ'.
    """
    paulis = ['I', 'X', 'Y', 'Z']
    phases = ['', 'i', '-', '-i']
    pauli_str = ''.join(numpy.random.choice(paulis) for _ in range(N))
    phase = numpy.random.choice(phases)
    return f'{phase}{pauli_str}'

def sample_question_and_answer(N):
    pauli_str1 = sample_random_pauli_string(N)
    pauli_str2 = sample_random_pauli_string(N)
    op1 = pc.pauli(pauli_str1)
    op2 = pc.pauli(pauli_str2)
    str1 = mcp.PauliTerm.from_obj(op1).text
    str2 = mcp.PauliTerm.from_obj(op2).text
    sol = mcp.PauliTerm.from_obj(op1 @ op2)
    return f'({str1}) ({str2})', sol

N_max = 10
iterations = 10


def generate_questions_and_answers(N_max, iterations):
    questions = []
    answers = []
    for i in range(iterations):
        q, a = sample_question_and_answer(N_max)
        questions.append(q)
        answers.append(a)

    questions_block = "\n".join(
        [f"({i+1}) Input: $ {q} $\n    Your Answer: " for i, q in enumerate(questions)]
    )

    pauli_instruction_template = f"""You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {{±1,±i}}. The Pauli operators obey the following multiplication rules:

    X_{{j}}^2 = Y_{{j}}^2 = Z_{{j}}^2 = I  
    X_{{j}}Y_{{j}} = iZ_{{j}}, Y_{{j}}Z_{{j}} = iX_{{j}}, Z_{{j}}X_{{j}} = iY_{{j}} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{{1}}Y_{{2}}Z_{{3}}), omitting identity operators. Do not include explanatory text or steps. Use LaTeX-style syntax, e.g., Z_{{1}}, X_{{2}} etc., and keep one space between operators.

    Examples:

    (1) Input: $(+iZ_{{1}})(−iX_{{1}})$  
        Answer: +iY_{{1}}

    (2) Input: $(+iZ_{{1}}Y_{{2}})(Z_{{1}}X_{{2}})$  
        Answer: Z_{{2}}

    Questions (Compute and give your answer in the same format as above):

    {questions_block}

    ---

    After answering all questions, output your answers as a Python list of strings for easy copying and checking.

    Please follow this exact format (note: this is an example, not related to the above questions):

    ```python
    LLM_answers = [
        '-i Y_{0}',
        '- X_{0} Z_{1}',
        'X_{0} Y_{1} Y_{2}',
        '- Y_{1} Z_{2} Y_{3}',
        '- Y_{0} X_{1} Y_{4}',
        '-i Z_{2} Z_{3} Z_{4} Z_{5} X_{6}',
        '+i Y_{0} Z_{1} Y_{2} X_{3} X_{6}',
        '+i Z_{0} Z_{1} Y_{2} Y_{3} Y_{4} Z_{6} Y_{8}',
        'Z_{1} X_{5} Z_{6} X_{7} Y_{8}',
        ...
    ]"""

    return pauli_instruction_template, answers

## Without MCP tools:

### ChatGPT

#### 4o model

In [2]:
N_max = 1
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [3]:
answers

[PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y'}, text='- Y_{0}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={}, text='- I'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z'}, text='+i Z_{0}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y'}, text='-i Y_{0}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y'}, text='-i Y_{0}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y'}, text='+i Y_{0}'),
 PauliTerm(coefficient=1j, pauli_string={}, text='+i I'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'X'}, text='- X_{0}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z'}, text='-i Z_{0}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y'}, text='+i Y_{0}')]

In [4]:
LLM_answers = [
    '+i Y_{0}',
    '-1',
    '+i Z_{0}',
    '+i Y_{0}',
    '-i Y_{0}',
    '+ Y_{0}',
    '+i',
    '+i X_{0}',
    '- Z_{0}',
    '+i Y_{0}'
]

In [5]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 1, question_volume  = 10, accuracy = 0.5


In [6]:
accuracy

[False, True, True, False, True, False, True, False, False, True]

________________________________________

In [7]:
N_max = 2
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [8]:
answers

[PauliTerm(coefficient=1j, pauli_string={0: 'X', 1: 'X'}, text='+i X_{0} X_{1}'),
 PauliTerm(coefficient=1j, pauli_string={1: 'X'}, text='+i X_{1}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z'}, text='- Z_{0}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y'}, text='+i Y_{0}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Z'}, text='-i Y_{0} Z_{1}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'Y'}, text='Y_{1}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'Z'}, text='+i Y_{0} Z_{1}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y'}, text='-i Y_{0}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={}, text='- I'),
 PauliTerm(coefficient=(-1+0j), pauli_string={1: 'Y'}, text='- Y_{1}')]

In [9]:
LLM_answers = [
    '+i X_{0} X_{1}',
    '+i X_{1}',
    '- Z_{0} Z_{1}',
    '+i Y_{0}',
    '+ Z_{0} Z_{1}',
    '- Y_{1}',
    '-i Y_{0} Z_{1}',
    '+i Y_{0}',
    '-1',
    '- Y_{1}'
]

In [10]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 2, question_volume  = 10, accuracy = 0.5


In [11]:
accuracy

[True, True, False, True, False, False, False, False, True, True]

________________________________________

In [12]:
N_max = 3
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [13]:
answers

[PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 2: 'Y'}, text='-i X_{0} Y_{2}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'X', 2: 'Y'}, text='X_{1} Y_{2}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'Z', 2: 'Y'}, text='-i X_{0} Z_{1} Y_{2}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'X', 1: 'Y', 2: 'Z'}, text='X_{0} Y_{1} Z_{2}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'Z', 2: 'X'}, text='Z_{1} X_{2}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y', 1: 'Z', 2: 'X'}, text='Y_{0} Z_{1} X_{2}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y', 1: 'Z', 2: 'X'}, text='Y_{0} Z_{1} X_{2}'),
 PauliTerm(coefficient=1j, pauli_string={1: 'Y', 2: 'Y'}, text='+i Y_{1} Y_{2}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 2: 'Y'}, text='-i X_{0} Y_{2}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'X', 2: 'X'}, text='+i Y_{0} X_{1} X_{2}')]

In [14]:
LLM_answers = [
    '- Y_{2} X_{0}',
    '+i Y_{1} Y_{2}',
    '+ Y_{0} Z_{1} Y_{2} Z_{0}',
    '+i X_{0} Y_{0} Y_{1} Z_{2}',
    '-i X_{1} X_{2}',
    '+i Y_{0} Z_{1} X_{2}',
    '+i Y_{0} Y_{1} X_{2}',
    '+i X_{1} Y_{1} Y_{2}',
    '-i X_{0} Y_{2}',
    '- Y_{0} X_{2}'
]

In [15]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 3, question_volume  = 10, accuracy = 0.2


In [16]:
accuracy

[False, False, False, False, False, False, False, True, True, False]

________________________________________

In [17]:
N_max = 4
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [18]:
answers

[PauliTerm(coefficient=(-1+0j), pauli_string={1: 'Z', 2: 'Z', 3: 'Z'}, text='- Z_{1} Z_{2} Z_{3}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'Y', 2: 'Z', 3: 'Z'}, text='- Y_{0} Y_{1} Z_{2} Z_{3}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 2: 'Z', 3: 'Y'}, text='+i Z_{0} Z_{2} Y_{3}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={1: 'Z', 2: 'Z', 3: 'X'}, text='-i Z_{1} Z_{2} X_{3}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'Y', 2: 'X', 3: 'X'}, text='+i Z_{0} Y_{1} X_{2} X_{3}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={2: 'X', 3: 'Z'}, text='-i X_{2} Z_{3}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'X', 2: 'Y', 3: 'Y'}, text='+i Z_{0} X_{1} Y_{2} Y_{3}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Y', 2: 'Z', 3: 'Y'}, text='-i Y_{0} Y_{1} Z_{2} Y_{3}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={1: 'Z', 3: 'X'}, text='-i Z_{1} X_{3}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z', 2: 'Y', 3: 'X'}, text='-i 

In [19]:
LLM_answers = [
    '+ Z_{1} Z_{2} Z_{3}',
    '-i X_{0} Y_{1} Z_{2} Z_{3}',
    '+ Z_{0} Z_{2} Y_{3}',
    '- Y_{1} Z_{2} X_{2}',
    '+i Z_{0} Z_{1} X_{2} Y_{3}',
    '- Y_{3} X_{2} X_{3}',
    '+i Z_{0} X_{2} Y_{3}',
    '+i X_{0} Y_{1} X_{3}',
    '- Z_{1}',
    '+ Z_{0} Y_{1} Y_{2} X_{0} X_{3}'
]

In [20]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 4, question_volume  = 10, accuracy = 0.0


In [21]:
accuracy

[False, False, False, False, False, False, False, False, False, False]

________________________________________

In [22]:
N_max = 5
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [23]:
answers

[PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y', 1: 'Y', 2: 'X', 3: 'X', 4: 'Z'}, text='Y_{0} Y_{1} X_{2} X_{3} Z_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z', 1: 'Z', 2: 'Y', 3: 'Z', 4: 'X'}, text='- Z_{0} Z_{1} Y_{2} Z_{3} X_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={1: 'X', 2: 'X', 3: 'Z', 4: 'Z'}, text='- X_{1} X_{2} Z_{3} Z_{4}'),
 PauliTerm(coefficient=(1+0j), pauli_string={2: 'X', 3: 'X'}, text='X_{2} X_{3}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 2: 'Z', 4: 'Z'}, text='- Y_{0} Z_{2} Z_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={1: 'X', 2: 'Z', 3: 'Z', 4: 'X'}, text='- X_{1} Z_{2} Z_{3} X_{4}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 1: 'Z', 3: 'X'}, text='+i X_{0} Z_{1} X_{3}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'Y', 4: 'X'}, text='+i Y_{0} Y_{1} X_{4}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'Y', 2: 'Y', 3: 'Z', 4: 'X'}, text='-i X_{0} Y_{1} Y_{2} Z_{3} X_{4}'),
 PauliTerm(coef

In [24]:
LLM_answers = [
    '+i Y_{0} Z_{1} X_{2} X_{3} Z_{4} X_{1}',
    '- Y_{1} Y_{2} Y_{3} Y_{4} Z_{0} X_{1} X_{3} Z_{4}',
    '+ Y_{3}',
    '+i Z_{2} X_{3} X_{4}',
    '+ Y_{0} Z_{2} Z_{4}',
    '- X_{1} X_{4} Z_{2} Z_{3}',
    '+ Y_{0} Z_{0} Z_{1} X_{1} X_{3}',
    '-i Y_{1} X_{4} Z_{0}',
    '+i Z_{0} Z_{1} Y_{2} Y_{4} Z_{3}',
    '- X_{0} X_{4}'
]

In [25]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 5, question_volume  = 10, accuracy = 0.0


In [26]:
accuracy

[False, False, False, False, False, False, False, False, False, False]

________________________________________

In [27]:
N_max = 6
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [28]:
answers

[PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Z', 2: 'Y', 3: 'Y', 4: 'Z'}, text='-i Y_{0} Z_{1} Y_{2} Y_{3} Z_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z', 1: 'Z', 3: 'X', 4: 'Z', 5: 'Y'}, text='- Z_{0} Z_{1} X_{3} Z_{4} Y_{5}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'Z', 2: 'Y', 3: 'X', 4: 'Z', 5: 'Z'}, text='+i Z_{0} Z_{1} Y_{2} X_{3} Z_{4} Z_{5}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'X', 2: 'Z', 3: 'X', 5: 'Y'}, text='+i Z_{0} X_{1} Z_{2} X_{3} Y_{5}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'X', 3: 'Z', 4: 'X'}, text='X_{1} Z_{3} X_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'X', 2: 'Z', 3: 'X', 4: 'Y', 5: 'X'}, text='- Y_{0} X_{1} Z_{2} X_{3} Y_{4} X_{5}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'X', 1: 'Z', 2: 'X', 5: 'Y'}, text='X_{0} Z_{1} X_{2} Y_{5}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 2: 'Z', 3: 'Y', 4: 'X', 5: 'X'}, text='+i Z_{0} Z_{2} Y_{3} X_{4} X_{5}'),
 PauliT

In [29]:
LLM_answers = [
    '+ Z_{4} Y_{0} Z_{1} Y_{2} Y_{3}',
    '- Z_{0} Z_{1} Y_{2} X_{3} Z_{4}',
    '- Z_{0} Z_{1} Y_{2} X_{3} Z_{4} Z_{5}',
    '- Z_{0} Y_{1} Z_{2} X_{3} Y_{5}',
    '- Z_{0} X_{1} X_{3} X_{4} Y_{3}',
    '+i Y_{0} X_{1} Z_{1} Z_{2} X_{3} Y_{4} X_{5}',
    '+i X_{0} Z_{1} X_{2} Y_{4}',
    '- X_{0} Z_{0} Z_{2} Y_{3} X_{4} Y_{5}',
    '+i Z_{0} Y_{1} Z_{3} X_{4} X_{5}',
    '- Z_{0} Z_{2} X_{2} Y_{3}'
]

In [30]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 6, question_volume  = 10, accuracy = 0.0


In [31]:
accuracy

[False, False, False, False, False, False, False, False, False, False]

________________________________________

In [32]:
N_max = 7
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [33]:
answers

[PauliTerm(coefficient=(1+0j), pauli_string={0: 'X', 2: 'Z', 3: 'Y', 6: 'X'}, text='X_{0} Z_{2} Y_{3} X_{6}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'Y', 2: 'Y', 3: 'Z', 4: 'X', 5: 'Y', 6: 'X'}, text='+i Y_{0} Y_{1} Y_{2} Z_{3} X_{4} Y_{5} X_{6}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'Z', 2: 'Z', 3: 'Z'}, text='+i Z_{0} Z_{1} Z_{2} Z_{3}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 2: 'Y', 3: 'X', 4: 'X', 5: 'Z'}, text='- Y_{0} Y_{2} X_{3} X_{4} Z_{5}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'Z', 2: 'X', 3: 'Y', 4: 'X', 5: 'Y', 6: 'Z'}, text='-i X_{0} Z_{1} X_{2} Y_{3} X_{4} Y_{5} Z_{6}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z', 1: 'X', 2: 'Z', 3: 'X', 4: 'Y', 6: 'Y'}, text='-i Z_{0} X_{1} Z_{2} X_{3} Y_{4} Y_{6}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 2: 'Z', 3: 'Y', 4: 'X', 5: 'Y'}, text='+i X_{0} Z_{2} Y_{3} X_{4} Y_{5}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={2: 'X', 3: 'X', 4: 'X', 5: 'Z

In [34]:
LLM_answers = [
    '+ X_{0} Z_{2} Y_{3} X_{6}',
    '+i Y_{0} Z_{3} X_{4} Y_{5}',
    '+i Z_{0} Z_{1} X_{2} Z_{3}',
    '- Z_{0} X_{0} X_{2} X_{3} X_{4} Z_{5}',
    '- Z_{0} Y_{0} Y_{1} X_{2} Y_{3} X_{5}',
    '+ Y_{3} Y_{4} Z_{6} Z_{0} X_{1}',
    '- X_{0} X_{2} Z_{3} X_{4} Y_{5}',
    '- Y_{2} X_{4} Y_{5} Z_{6}',
    '- Z_{0} X_{1} X_{4} Y_{6}',
    '+ Z_{0} X_{0} Y_{1} X_{1} Y_{4} Z_{5} Y_{6}'
]

In [35]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 7, question_volume  = 10, accuracy = 0.1


In [36]:
accuracy

[True, False, False, False, False, False, False, False, False, False]

________________________________________

In [37]:
N_max = 8
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [38]:
answers

[PauliTerm(coefficient=(1+0j), pauli_string={0: 'X', 1: 'X', 2: 'X', 3: 'Z', 5: 'X', 6: 'Y', 7: 'X'}, text='X_{0} X_{1} X_{2} Z_{3} X_{5} Y_{6} X_{7}'),
 PauliTerm(coefficient=(1+0j), pauli_string={3: 'X', 6: 'Z'}, text='X_{3} Z_{6}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 1: 'Y', 2: 'Y', 3: 'X', 4: 'Y', 5: 'Z', 6: 'Y', 7: 'Z'}, text='+i X_{0} Y_{1} Y_{2} X_{3} Y_{4} Z_{5} Y_{6} Z_{7}'),
 PauliTerm(coefficient=(1+0j), pauli_string={2: 'Z', 4: 'Z', 5: 'Y', 6: 'Y', 7: 'Y'}, text='Z_{2} Z_{4} Y_{5} Y_{6} Y_{7}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'X', 2: 'Y', 3: 'Z', 4: 'Z', 5: 'Y', 6: 'Y', 7: 'Z'}, text='- Y_{0} X_{1} Y_{2} Z_{3} Z_{4} Y_{5} Y_{6} Z_{7}'),
 PauliTerm(coefficient=1j, pauli_string={1: 'X', 2: 'X', 3: 'Y', 4: 'Y', 5: 'Y'}, text='+i X_{1} X_{2} Y_{3} Y_{4} Y_{5}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y', 1: 'Y', 4: 'X', 6: 'Z'}, text='Y_{0} Y_{1} X_{4} Z_{6}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={1: 'X', 4: 'X', 5: 'Y'

In [39]:
LLM_answers = [
    '+i X_{0} X_{1} X_{2} Z_{3} X_{5} Y_{6} X_{7}',
    '-i X_{3}',
    '- Y_{0} Y_{1} Y_{2} X_{3} Y_{4} Z_{5} Z_{6}',
    '- X_{2} Z_{4} X_{5} Z_{5} Y_{6}',
    '- Y_{0} X_{1} X_{2} Z_{3} Z_{5} Y_{5} Z_{6}',
    '+i X_{2} Y_{3} Y_{4} Y_{5}',
    '- X_{0} X_{1} Z_{4} Z_{6}',
    '- Y_{1} X_{4} Y_{5}',
    '+i X_{0} Z_{0} Y_{1} Y_{3} Z_{4} X_{5} Z_{6}',
    '- Y_{0} X_{3} Z_{5} X_{7}'
]

In [40]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 8, question_volume  = 10, accuracy = 0.0


________________________________________

In [41]:
N_max = 9
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [42]:
answers

[PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'Z', 3: 'Z', 5: 'Y', 6: 'X', 8: 'Y'}, text='-i X_{0} Z_{1} Z_{3} Y_{5} X_{6} Y_{8}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z', 1: 'X', 2: 'Z', 3: 'Y', 4: 'Z', 6: 'Y', 7: 'Z', 8: 'Y'}, text='Z_{0} X_{1} Z_{2} Y_{3} Z_{4} Y_{6} Z_{7} Y_{8}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'X', 2: 'Z', 3: 'Y', 6: 'Y', 7: 'Z'}, text='- Y_{0} X_{1} Z_{2} Y_{3} Y_{6} Z_{7}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={1: 'Z', 2: 'Z', 3: 'X', 5: 'Y', 6: 'X', 7: 'X', 8: 'Z'}, text='-i Z_{1} Z_{2} X_{3} Y_{5} X_{6} X_{7} Z_{8}'),
 PauliTerm(coefficient=1j, pauli_string={1: 'Z', 3: 'Z', 5: 'Z', 8: 'Y'}, text='+i Z_{1} Z_{3} Z_{5} Y_{8}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'X', 1: 'Z', 2: 'Y', 3: 'Z', 4: 'Y', 5: 'Y', 6: 'Y', 7: 'Y', 8: 'Y'}, text='- X_{0} Z_{1} Y_{2} Z_{3} Y_{4} Y_{5} Y_{6} Y_{7} Y_{8}'),
 PauliTerm(coefficient=1j, pauli_string={1: 'Z', 3: 'X', 6: 'Y'}, text='+i Z_{1} X_{3} Y_{6}'),
 

In [43]:
LLM_answers = [
    '+ Y_{0} Z_{0} Z_{1} Z_{3} Y_{5} Y_{6} Z_{8}',
    '+ Z_{0} X_{1} Y_{3} Z_{4} Z_{6} Z_{7}',
    '- X_{0} X_{1} Y_{2} Y_{3} Z_{5} Y_{6} Z_{7}',
    '-i Z_{1} X_{2} X_{3} Y_{2} Z_{4} Y_{5} X_{6} X_{7} Z_{8}',
    '- X_{1} Z_{3} Y_{5} Y_{8}',
    '- Y_{0} Z_{1} Y_{2} Z_{3} Z_{5} Z_{6} Y_{7} Y_{8}',
    '- X_{1} X_{3} Z_{6}',
    '+ Y_{0} X_{1} Z_{2} X_{4} X_{5} Z_{6} Z_{7}',
    '+i X_{1} Z_{0} Y_{2} Y_{4} Y_{5} X_{7} Z_{8}',
    '+ Y_{0} X_{1} X_{3} Y_{4} Y_{5} Z_{6} Y_{7}'
]

In [44]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 9, question_volume  = 10, accuracy = 0.0


________________________________________

In [45]:
N_max = 10
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [46]:
answers

[PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Z', 3: 'X', 4: 'Y', 6: 'Z', 7: 'Z', 8: 'Z', 9: 'Z'}, text='-i Y_{0} Z_{1} X_{3} Y_{4} Z_{6} Z_{7} Z_{8} Z_{9}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'X', 4: 'X', 5: 'Z', 6: 'Z', 7: 'Y', 8: 'X'}, text='X_{1} X_{4} Z_{5} Z_{6} Y_{7} X_{8}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'X', 2: 'Z', 3: 'Z', 5: 'Z', 6: 'Z', 7: 'X', 8: 'X', 9: 'X'}, text='+i Z_{0} X_{1} Z_{2} Z_{3} Z_{5} Z_{6} X_{7} X_{8} X_{9}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z', 1: 'Z', 2: 'X', 3: 'Z', 4: 'Y', 6: 'Z', 7: 'Y', 8: 'Z', 9: 'Y'}, text='- Z_{0} Z_{1} X_{2} Z_{3} Y_{4} Z_{6} Y_{7} Z_{8} Y_{9}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'Z', 2: 'X', 3: 'Y', 4: 'Z', 5: 'X', 6: 'Y', 7: 'Z', 8: 'Y', 9: 'Z'}, text='Z_{1} X_{2} Y_{3} Z_{4} X_{5} Y_{6} Z_{7} Y_{8} Z_{9}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={1: 'Y', 2: 'X', 3: 'Z', 5: 'X', 6: 'Y', 7: 'Y', 8: 'Z', 9: 'Y'}, text='- Y_{1} X_{2} Z_{3} X_{5} Y_

In [47]:
LLM_answers = [
    '+i Y_{0} X_{1} Y_{3} Y_{4} Y_{5} Y_{6} Z_{7} Z_{8} Z_{9}',
    '+ Z_{1} Z_{4} Z_{5} Z_{6} Z_{7} Z_{8}',
    '+ Z_{0} X_{1} Z_{1} X_{2} Z_{3} Z_{4} Z_{6} Z_{7} X_{8}',
    '- Z_{0} Z_{1} X_{2} Z_{3} Y_{4} Z_{5} Z_{6} Y_{8} Y_{9}',
    '- Y_{1} X_{2} Y_{3} Z_{4} X_{6} X_{9}',
    '+i Z_{1} X_{2} X_{3} X_{5} Z_{6} Z_{7} Z_{8}',
    '+i Y_{0} Y_{2} X_{4} Y_{5} Z_{8} Z_{9}',
    '+ X_{0} Z_{2} X_{3} Z_{3} Y_{5} Z_{7} Z_{9} Y_{9}',
    '- X_{0} Y_{1} X_{1} Z_{2} Y_{3} X_{4} X_{5} Z_{5}',
    '- X_{0} Z_{0} Y_{1} X_{2} X_{3} X_{9}'
]

In [48]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 10, question_volume  = 10, accuracy = 0.0


________________________________________

In [2]:
N_max = 20
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [3]:
LLM_answers = [
    '+i Z_0 Y_1 Y_2 Y_3 X_5 Z_6 X_7 Z_8 Y_9 Z_10 Z_11 X_13 X_14 Z_15 Z_16 Z_17 Z_18 Z_19',
    '-i X_0 Y_1 I_2 Y_3 Z_4 Y_5 Z_6 Y_7 Z_8 X_9 Z_10 Z_11 Z_12 I_13 I_14 X_15 X_16 X_18 X_19',
    '- Z_0 Y_1 Y_2 I_3 Y_4 Z_5 Y_6 Z_7 X_8 Z_9 X_10 Z_11 Y_12 I_13 I_14 X_15 Y_16 Z_17 Z_18 X_19',
    '-i X_0 X_1 I_2 I_3 Y_4 Y_5 Z_6 Y_7 Z_8 Y_9 Z_10 Z_11 Y_12 Z_13 I_14 Z_15 Y_17 Y_18 Y_19',
    '- Z_0 Z_1 Z_3 Z_4 Z_5 Y_6 Y_7 X_8 X_9 Y_10 Y_11 Y_12 Y_13 X_14 X_15 X_16 Z_17 Z_18 X_19',
    '+i X_0 Y_1 X_2 X_3 X_4 Z_5 Y_7 Y_8 Y_9 X_10 Z_11 X_12 X_13 X_14 Z_15 X_16 Y_17 Z_19',
    '+i Y_0 Y_1 Y_2 I_3 Y_4 Z_5 Y_6 Y_7 X_8 I_9 X_10 Y_11 Z_12 X_13 X_14 I_15 Y_16 Y_17 Y_18 Z_19',
    '+i Y_0 Z_1 Z_2 I_3 I_4 X_5 Z_6 Z_7 I_8 X_9 Z_10 Y_11 Z_12 I_13 I_14 X_15 X_16 Z_17 Z_18 Z_19',
    '+i Y_0 X_1 Y_2 I_3 I_4 Z_5 X_6 X_7 X_8 X_9 Y_10 X_11 Z_12 X_13 X_14 I_15 Y_16 Z_17 I_18',
    '-i Y_0 Y_1 Y_2 X_4 Z_5 Y_6 X_7 Y_8 Z_9 X_10 X_11 X_12 X_13 X_14 I_15 I_17 I_18 I_19'
]

In [4]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 20, question_volume  = 10, accuracy = 0.0


________________________________________

In [5]:
N_max = 30
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [6]:
LLM_answers = [
    '+i X_{0} Y_{1} X_{2} Y_{3} X_{4} Y_{5} Y_{6} Z_{7} Z_{8} Z_{9} Z_{10} X_{11} Z_{12} Z_{14} Z_{15} Y_{16} Y_{17} Z_{18} Z_{19} Y_{20} Z_{21} Z_{22} Y_{23} Z_{24} Z_{25} Z_{26} Z_{27} Z_{28} Z_{29}',
    '+i Z_{0} Z_{1} X_{2} X_{3} Z_{4} Z_{5} Z_{6} Y_{7} Z_{8} Z_{9} Y_{10} Z_{11} Z_{12} Z_{13} Z_{14} Y_{15} Z_{16} Z_{17} Z_{18} Z_{19} Z_{20} Z_{21} Z_{22} Z_{23} Z_{24} Z_{25} Z_{26} Z_{27} Z_{28} Z_{29}',
    '+i Y_{0} Z_{1} X_{2} Z_{3} Z_{4} Z_{5} Z_{6} Z_{7} Z_{8} Z_{9} Z_{10} Z_{11} Z_{12} Z_{13} Z_{14} Z_{15} Z_{16} Z_{17} Z_{18} Z_{19} Z_{20} Z_{21} Z_{22} Z_{23} Z_{24} Z_{25} Z_{26} Z_{27} Z_{28} Z_{29}',
    '+i Y_{0} Z_{1} Z_{2} Z_{3} Z_{4} Z_{5} Z_{6} Z_{7} Z_{8} Z_{9} Z_{10} Z_{11} Z_{12} Z_{13} Z_{14} Z_{15} Z_{16} Z_{17} Z_{18} Z_{19} Z_{20} Z_{21} Z_{22} Z_{23} Z_{24} Z_{25} Z_{26} Z_{27} Z_{28} Z_{29}',
    '+i Z_{0} Z_{1} Z_{2} Z_{3} Z_{4} Z_{5} Z_{6} Z_{7} Z_{8} Z_{9} Z_{10} Z_{11} Z_{12} Z_{13} Z_{14} Z_{15} Z_{16} Z_{17} Z_{18} Z_{19} Z_{20} Z_{21} Z_{22} Z_{23} Z_{24} Z_{25} Z_{26} Z_{27} Z_{28} Z_{29}',
    '+i Z_{0} Z_{1} Z_{2} Z_{3} Z_{4} Z_{5} Z_{6} Z_{7} Z_{8} Z_{9} Z_{10} Z_{11} Z_{12} Z_{13} Z_{14} Z_{15} Z_{16} Z_{17} Z_{18} Z_{19} Z_{20} Z_{21} Z_{22} Z_{23} Z_{24} Z_{25} Z_{26} Z_{27} Z_{28} Z_{29}',
    '+i Z_{0} Z_{1} Z_{2} Z_{3} Z_{4} Z_{5} Z_{6} Z_{7} Z_{8} Z_{9} Z_{10} Z_{11} Z_{12} Z_{13} Z_{14} Z_{15} Z_{16} Z_{17} Z_{18} Z_{19} Z_{20} Z_{21} Z_{22} Z_{23} Z_{24} Z_{25} Z_{26} Z_{27} Z_{28} Z_{29}',
    '+i Z_{0} Z_{1} Z_{2} Z_{3} Z_{4} Z_{5} Z_{6} Z_{7} Z_{8} Z_{9} Z_{10} Z_{11} Z_{12} Z_{13} Z_{14} Z_{15} Z_{16} Z_{17} Z_{18} Z_{19} Z_{20} Z_{21} Z_{22} Z_{23} Z_{24} Z_{25} Z_{26} Z_{27} Z_{28} Z_{29}',
    '+i Z_{0} Z_{1} Z_{2} Z_{3} Z_{4} Z_{5} Z_{6} Z_{7} Z_{8} Z_{9} Z_{10} Z_{11} Z_{12} Z_{13} Z_{14} Z_{15} Z_{16} Z_{17} Z_{18} Z_{19} Z_{20} Z_{21} Z_{22} Z_{23} Z_{24} Z_{25} Z_{26} Z_{27} Z_{28} Z_{29}',
    '+i Z_{0} Z_{1} Z_{2} Z_{3} Z_{4} Z_{5} Z_{6} Z_{7} Z_{8} Z_{9} Z_{10} Z_{11} Z_{12} Z_{13} Z_{14} Z_{15} Z_{16} Z_{17} Z_{18} Z_{19} Z_{20} Z_{21} Z_{22} Z_{23} Z_{24} Z_{25} Z_{26} Z_{27} Z_{28} Z_{29}'
]

In [7]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 30, question_volume  = 10, accuracy = 0.0
